# Semi-Supervised Learning: FixMatch, Mean Teacher, and nnPU Learning

This notebook explores foundational algorithms in semi-supervised learning (SSL) and positive-unlabeled (PU) learning:
1. **FixMatch**: Weak-to-strong augmentation consistency with hard pseudo-label confidence thresholding.
2. **Mean Teacher**: Temporal ensembling via exponential moving average (EMA) teacher networks.
3. **MixMatch**: Label guessing and entropy minimization via distribution sharpening.
4. **nnPU Learning**: Non-Negative empirical risk bounds preventing overfitting when only positive and unlabeled data are observed.

**Decision Boundary (When NOT to Use)**:
- Do not use FixMatch when strong data augmentations (RandAugment, Cutout) destroy class semantics (e.g., medical pathology where color inversion or crop removes critical lesions).
- Do not use standard unbiased PU (`uPU`) without the non-negative clamp ($\max(0, \cdot)$) when training flexible deep models, as empirical negative risk rapidly diverges to $-\infty$.

**Topic Bridges & Connects To**:
- `02_non_contrastive_and_masked_ssl_byol_dino_vicreg_mae_jepa` (Unsupervised representation learning)
- `04_meta_learning_prototypical_maml_and_reptile` (Few-shot and adaptation paradigms)

## 2. Mathematical Formulations & Theoretical Foundations

- **FixMatch Thresholded Cross-Entropy**:
  $$\mathcal{L}_u = \frac{1}{B} \sum_{b=1}^B \mathbb{I}(\max(q_b) \ge \tau) \cdot H(\hat{y}_b, p_m(y \mid \mathcal{A}(u_b))), \quad \mathcal{O}(B \cdot K)$$
- **Mean Teacher Consistency MSE**:
  $$\mathcal{L}_{\text{consistency}} = \frac{1}{B} \sum_{i=1}^B \| \sigma(f_\theta(x_i)) - \sigma(f_{\theta'}(x_i)) \|_2^2, \quad \mathcal{O}(B \cdot K)$$
- **MixMatch Temperature Sharpening**:
  $$\text{Sharpen}(p, T)_k = \frac{p_k^{1/T}}{\sum_j p_j^{1/T}}, \quad \mathcal{O}(B \cdot K)$$
- **Non-Negative Positive-Unlabeled (nnPU) Risk**:
  $$\tilde{R}_{\text{nnPU}}(g) = \pi_p \hat{R}_p^+(g) + \max\left(0, \hat{R}_u^-(g) - \pi_p \hat{R}_p^-(g)\right), \quad \mathcal{O}(N_p + N_u)$$

In [ ]:
import math
from pathlib import Path
import torch
import torch.nn.functional as F
from safetensors.torch import load_file

torch.manual_seed(42)
REPO_ROOT = next(p for p in Path.cwd().parents if (p / "Cargo.toml").exists()) if not (Path.cwd() / "Cargo.toml").exists() else Path.cwd()
GOLDEN_PATH = REPO_ROOT / "weights" / "08_learning_paradigms_rl_and_alignment" / "01_self_semi_meta_continual_federated" / "03_semi_supervised_fixmatch_mean_teacher_and_nnpu_learning.safetensors"
golden = load_file(str(GOLDEN_PATH))
print(f"Loaded {len(golden)} tensors from golden checkpoint.")

In [ ]:
# 1. Mean Teacher Consistency Loss
def mean_teacher_consistency_loss(student_logits: torch.Tensor, teacher_logits: torch.Tensor) -> torch.Tensor:
    p_s = F.softmax(student_logits, dim=-1)
    p_t = F.softmax(teacher_logits, dim=-1)
    return F.mse_loss(p_s, p_t).unsqueeze(0)

mt_calc = mean_teacher_consistency_loss(golden["student_logits"], golden["teacher_logits"])
diff_mt = torch.abs(mt_calc - golden["mt_loss_ref"]).item()
print(f"Mean Teacher Consistency max abs error: {diff_mt:.2e}")
assert diff_mt < 1e-5

In [ ]:
# 2. FixMatch Confidence-Thresholded Consistency Loss
def fixmatch_thresholded_loss(
    weak_logits: torch.Tensor,
    strong_logits: torch.Tensor,
    tau: float = 0.6,
) -> torch.Tensor:
    q = F.softmax(weak_logits, dim=-1)
    max_prob, pseudo_label = q.max(dim=-1)
    mask = (max_prob >= tau).float()
    ce = F.cross_entropy(strong_logits, pseudo_label, reduction="none")
    return (mask * ce).mean().unsqueeze(0)

fm_calc = fixmatch_thresholded_loss(golden["weak_logits"], golden["strong_logits"], tau=0.6)
diff_fm = torch.abs(fm_calc - golden["fixmatch_loss_ref"]).item()
print(f"FixMatch Thresholded Loss max abs error: {diff_fm:.2e}")
assert diff_fm < 1e-5

In [ ]:
# 3. MixMatch Distribution Sharpening
def mixmatch_sharpen(probs: torch.Tensor, temperature: float = 0.5) -> torch.Tensor:
    sharpened = probs ** (1.0 / temperature)
    return sharpened / sharpened.sum(dim=-1, keepdim=True)

mm_calc = mixmatch_sharpen(golden["p_unsharpened"], temperature=0.5)
diff_mm = torch.max(torch.abs(mm_calc - golden["mixmatch_sharpen_ref"])).item()
print(f"MixMatch Sharpening max abs error: {diff_mm:.2e}")
assert diff_mm < 1e-5

In [ ]:
# 4. Non-Negative Positive-Unlabeled (nnPU) Risk & Anti-Pattern Contrast
def nnpu_risk(
    g_pos: torch.Tensor,
    g_unlabeled: torch.Tensor,
    pi_p: float = 0.3,
) -> torch.Tensor:
    ell_p_pos = F.softplus(-g_pos).mean()
    ell_p_neg = F.softplus(g_pos).mean()
    ell_u_neg = F.softplus(g_unlabeled).mean()
    neg_risk = ell_u_neg - pi_p * ell_p_neg
    return (pi_p * ell_p_pos + torch.clamp(neg_risk, min=0.0)).unsqueeze(0)

nnpu_calc = nnpu_risk(golden["g_pos"], golden["g_unlabeled"], pi_p=0.3)
diff_nnpu = torch.abs(nnpu_calc - golden["nnpu_risk_ref"]).item()
assert diff_nnpu < 1e-5

# Misuse Anti-Pattern: Unclamped uPU allowing negative empirical risk
# Synthesizing extreme overconfident positive classifier values
g_overfit_pos = torch.full((10,), 15.0)
g_unlabeled_neutral = torch.zeros(10)
pi = 0.5
ell_p_neg_bad = F.softplus(g_overfit_pos).mean() # ~15.0
ell_u_neg_bad = F.softplus(g_unlabeled_neutral).mean() # ~0.693
unconstrained_neg_risk = ell_u_neg_bad - pi * ell_p_neg_bad
# In uPU this drifts deeply negative (-6.8) causing catastrophic overfit!
assert unconstrained_neg_risk.item() < 0.0, "Anti-Pattern: uPU risk must become negative under extreme predictions"
# In nnPU, clamping guarantees strictly non-negative risk:
clamped_neg_risk = torch.clamp(unconstrained_neg_risk, min=0.0)
assert clamped_neg_risk.item() == 0.0, "nnPU Failure: Risk must be clamped to 0"

print("=" * 65)
print(f"{'Operator':<32} | {'Max Abs Error':<15} | Status")
print("-" * 65)
print(f"{'Mean Teacher Consistency':<32} | {diff_mt:<15.2e} | PASSED")
print(f"{'FixMatch Thresholded Loss':<32} | {diff_fm:<15.2e} | PASSED")
print(f"{'MixMatch Sharpening':<32} | {diff_mm:<15.2e} | PASSED")
print(f"{'nnPU Non-Negative Risk':<32} | {diff_nnpu:<15.2e} | PASSED")
print("=" * 65)

## 8. Summary, Key Takeaways & Cross-Domain Generalisation

- [x] **FixMatch**: Dynamic thresholding $\tau$ completely prevents confirmation bias on noisy low-confidence unlabeled samples.
- [x] **Mean Teacher**: EMA temporal weight averaging creates an ensemble teacher yielding reliable smooth consistency signals.
- [x] **MixMatch**: Distribution sharpening enforces the cluster assumption by minimizing prediction entropy.
- [x] **nnPU**: Enforcing non-negative negative risk bounds ($\max(0, \hat{R}_n^-)$) allows training robust binary classifiers from positive and unlabeled data alone.

**Cross-Domain Generalisation**:
Semi-supervised and PU paradigms generalize from standard image recognition to genomics (disease-associated gene discovery where unassociated genes are unlabeled), cybersecurity (intrusion detection where only confirmed attacks are positive), and audio transcription (leveraging petabytes of uncurated speech).

See curriculum index: [INDEX_BY_GOAL](../../../INDEX_BY_GOAL.md).